# WildJailbreak — Phase 1 single-turn generalization (Llama-3.1-8B)

Tests whether the JBB-trained single-turn harmfulness direction `v_harmful(ST)`
generalizes to a different prompt distribution and survives single-turn adversarial
framing. Extraction + analysis, Llama only.

| Test | Pair | Question |
|---|---|---|
| A | harmful_vanilla vs benign_vanilla | Does the JBB direction transfer to a new prompt distribution? |
| B | harmful_adversarial vs benign_adversarial | Is the direction robust under single-turn adversarial framing? |
| C | harmful_vanilla vs harmful_adversarial | Does adversarial framing *already* displace the representation in single-turn? |
| D | benign_adversarial vs benign_vanilla | Does the model encode "adversarial-looking" independent of harmfulness? |

If A ≥ 0.9 and B ≥ 0.9 → direction generalizes, fine to invest in multi-turn WJB (Phase 2).
If A is fine but B drops sharply → adversarial framing *alone* displaces the direction in
single-turn, and multi-turn displacement is (partly) a stacked version of that effect.
If C is large → the model separates vanilla from adversarial phrasing along `v_harmful`,
which would be a confounder to understand before extending.

WildJailbreak is gated. Before running, accept at
https://huggingface.co/datasets/allenai/wildjailbreak and authenticate via
`huggingface-cli login` or `HF_TOKEN`.

## 1. Setup

In [ ]:
import os
# Set visible GPUs before importing torch.
GPU_IDS = [4, 5, 6]
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(str(g) for g in GPU_IDS)
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import roc_auc_score

repo_root = Path('../..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.extraction.runner import run_single_turn_baseline

MODEL       = 'llama'
MODEL_ID    = 'meta-llama/Llama-3.1-8B-Instruct'
DTYPE       = torch.bfloat16
USER_END_TOKEN = '<|eot_id|>'

N_TRANSFORMER_LAYERS_TOTAL = 32
LAYER_INDICES = (np.linspace(0, N_TRANSFORMER_LAYERS_TOTAL - 1, 8, dtype=int) + 1).tolist()

N_PER_SUBSET = 500          # samples per data_type subset
RNG_SEED     = 42

SHARED_SYSTEM_PROMPT = (
    "You are a helpful, respectful and honest assistant. Always answer as helpfully "
    "as possible, while being safe.\n\nYour answers should not include any harmful, "
    "unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure "
    "that your responses are socially unbiased and positive in nature.\n\nIf a "
    "question does not make any sense, or is not factually coherent, explain why "
    "instead of answering something not correct. If you don't know the answer to a "
    "question, please don't share false information."
)

REPR_ROOT = repo_root / 'data' / MODEL / 'representations'
WJB_ROOT  = REPR_ROOT / 'wildjailbreak'
FIG_DIR   = repo_root / 'figures'
FIG_DIR.mkdir(exist_ok=True)

DATA_TYPES = ['harmful_vanilla', 'harmful_adversarial', 'benign_vanilla', 'benign_adversarial']

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})
print(f'REPR_ROOT: {REPR_ROOT}')
print(f'WJB_ROOT : {WJB_ROOT}')
print(f'Layers   : {LAYER_INDICES} (of {N_TRANSFORMER_LAYERS_TOTAL})')
print(f'N/subset : {N_PER_SUBSET}')


## 2. Load and sample WildJailbreak

Random sample without replacement per `data_type`. For `*_vanilla` rows we use
the `vanilla` column; for `*_adversarial` rows we use the `adversarial` column
(which is the actually-adversarial single-turn prompt).

In [ ]:
ds = load_dataset(
    'allenai/wildjailbreak', 'train',
    delimiter='\t', keep_default_na=False,
)
df = ds['train'].to_pandas()
print(f"WJB train: {len(df):,} rows, counts by data_type:")
print(df['data_type'].value_counts())

rng = np.random.default_rng(RNG_SEED)
samples = {}
for dt in DATA_TYPES:
    sub = df[df['data_type'] == dt].reset_index(drop=True)
    n   = min(N_PER_SUBSET, len(sub))
    idx = rng.choice(len(sub), size=n, replace=False)
    use_col = 'adversarial' if dt.endswith('adversarial') else 'vanilla'
    goals = sub.loc[idx, use_col].tolist()
    samples[dt] = pd.DataFrame({
        'Index':    np.arange(n),
        'Goal':     goals,
        'Behavior': [g[:80] for g in goals],
        'Category': dt,
    })
    print(f'  {dt:22s} kept {n} samples from column {use_col!r}')


## 3. Extract hidden states

One call to `run_single_turn_baseline` per `data_type`. Saves arrays shaped
`(N, 8, 4096)` float16 plus `metadata.parquet` and `layer_indices.json` into
`data/llama/representations/wildjailbreak/{data_type}/`. Resume-safe: existing
folders are skipped.

In [ ]:
for dt in DATA_TYPES:
    save_dir = WJB_ROOT / dt
    if (save_dir / 'metadata.parquet').exists():
        print(f'skip {save_dir} (already extracted)')
        continue
    run_single_turn_baseline(
        goal_type=dt,
        goals_df=samples[dt],
        system_prompt=SHARED_SYSTEM_PROMPT,
        save_dir=save_dir,
        model_id=MODEL_ID,
        logical_gpu_ids=LOGICAL_GPU_IDS,
        layer_indices=LAYER_INDICES,
        dtype=DTYPE,
        user_end_token=USER_END_TOKEN,
        n_transformer_layers_total=N_TRANSFORMER_LAYERS_TOTAL,
    )


## 4. Load representations — WJB subsets and the JBB single-turn anchor

In [ ]:
def load_repr(folder):
    meta   = pd.read_parquet(folder / 'metadata.parquet')
    h_inst = np.load(folder / 'h_inst.npy')
    h_post = np.load(folder / 'h_post_inst.npy')
    assert len(meta) == len(h_inst) == len(h_post)
    return {'meta': meta, 'h_inst': h_inst, 'h_post': h_post}

wjb = {}
for dt in DATA_TYPES:
    folder = WJB_ROOT / dt
    if not folder.exists():
        print(f'  MISSING: {folder}')
        continue
    wjb[dt] = load_repr(folder)
    print(f'  {dt:22s} n={len(wjb[dt]["meta"]):4d}  h_inst={wjb[dt]["h_inst"].shape}')

# JBB single-turn anchor (paired 100 harmful / 100 benign)
jbb_st = {
    'harmful': load_repr(REPR_ROOT / 'single_turn' / 'harmful'),
    'benign':  load_repr(REPR_ROOT / 'single_turn' / 'benign'),
}
print()
print(f"JBB single_turn harmful: {jbb_st['harmful']['h_inst'].shape}")
print(f"JBB single_turn benign : {jbb_st['benign']['h_inst'].shape}")

# Layer labels (should match across all folders)
layer_info = json.loads((WJB_ROOT / DATA_TYPES[0] / 'layer_indices.json').read_text())
LAYER_LABELS = layer_info['labels']
N_SAVED = layer_info['n_sweep']
print(f"\nLayer sweep: {LAYER_LABELS}")


## 5. Compute the JBB-trained `v_harmful(ST)`

Within-pair subtraction on the 100 JBB pairs at each of the 8 saved layers.
Produces a (8, 4096) unit-normalised direction. Separate computations for
`h_inst` and `h_post`.

In [ ]:
def v_harmful_st(jbb, h_key):
    m_h = jbb['harmful']['meta'].assign(_idx=lambda x: x.index).sort_values('pair_id')
    m_b = jbb['benign']['meta'].assign(_idx=lambda x: x.index).sort_values('pair_id')
    common = sorted(set(m_h['pair_id']) & set(m_b['pair_id']))
    idx_h  = [int(m_h.loc[m_h['pair_id'] == p, '_idx'].iloc[0]) for p in common]
    idx_b  = [int(m_b.loc[m_b['pair_id'] == p, '_idx'].iloc[0]) for p in common]
    diffs  = jbb['harmful'][h_key][idx_h].astype(np.float32) - jbb['benign'][h_key][idx_b].astype(np.float32)
    d      = diffs.mean(0)
    return d / (np.linalg.norm(d, axis=-1, keepdims=True) + 1e-12)

v_st_inst = v_harmful_st(jbb_st, 'h_inst')
v_st_post = v_harmful_st(jbb_st, 'h_post')
print(f'v_st_inst: {v_st_inst.shape}   v_st_post: {v_st_post.shape}')


## 6. Four AUROC tests at every layer

For each (test, layer), concat the two subsets' hidden states, project onto
`v_harmful(ST)` at that layer, and compute AUROC against the binary "group-A
vs group-B" label. `group_a=1` is always the "more harmful / more adversarial"
side; higher projection should give AUROC > 0.5 if `v_harmful(ST)` discriminates.

In [ ]:
TESTS = [
    ('A',  'harmful_vanilla',     'benign_vanilla',      'Transfer (vanilla)'),
    ('B',  'harmful_adversarial', 'benign_adversarial',  'Robustness (adversarial)'),
    ('C',  'harmful_adversarial', 'harmful_vanilla',     'Adv. framing displaces?'),
    ('D',  'benign_adversarial',  'benign_vanilla',      'Adv. framing alone?'),
]

def auroc_at_layer(h_a, h_b, v, L):
    '''Project (h_a, h_b) at layer L onto v[L], compute AUROC with a=1, b=0.'''
    proj_a = h_a[:, L, :].astype(np.float32) @ v[L]
    proj_b = h_b[:, L, :].astype(np.float32) @ v[L]
    y = np.concatenate([np.ones(len(proj_a)), np.zeros(len(proj_b))])
    s = np.concatenate([proj_a, proj_b])
    return roc_auc_score(y, s)

rows = []
for pos, v_st, pos_label in [('h_inst', v_st_inst, 't_inst'),
                              ('h_post', v_st_post, 't_post')]:
    for test_id, a_key, b_key, desc in TESTS:
        if a_key not in wjb or b_key not in wjb:
            continue
        h_a = wjb[a_key][pos];  h_b = wjb[b_key][pos]
        for L in range(N_SAVED):
            rows.append({
                'position':  pos_label,
                'test':      test_id,
                'desc':      desc,
                'layer_pos': L,
                'layer':     LAYER_LABELS[L],
                'auroc':     auroc_at_layer(h_a, h_b, v_st, L),
                'n_a':       len(h_a),
                'n_b':       len(h_b),
            })
auroc_df = pd.DataFrame(rows)
auroc_df.head()


### 6a. Line plot — AUROC across layers, one panel per token position

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
colors  = {'A': '#1f77b4', 'B': '#d62728', 'C': '#2ca02c', 'D': '#9467bd'}
markers = {'A': 'o', 'B': 's', 'C': '^', 'D': 'x'}

for ax, pos in zip(axes, ['t_inst', 't_post']):
    sub = auroc_df[auroc_df['position'] == pos]
    for test_id, a_key, b_key, desc in TESTS:
        s = sub[sub['test'] == test_id].sort_values('layer_pos')
        if s.empty:
            continue
        ax.plot(s['layer_pos'], s['auroc'], marker=markers[test_id], color=colors[test_id],
                lw=1.8, label=f'{test_id} — {desc}')
    ax.axhline(0.5, color='gray', ls=':', lw=0.8)
    ax.axhline(0.9, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set_xticks(range(N_SAVED));  ax.set_xticklabels(LAYER_LABELS, fontsize=8)
    ax.set_xlabel('layer');  ax.set_ylim(0.3, 1.02);  ax.grid(alpha=0.3)
    ax.set_title(f'position = {pos}')
    if ax is axes[0]:
        ax.set_ylabel('AUROC (projection onto JBB-trained v_harmful(ST))')
    ax.legend(fontsize=8, loc='lower left', frameon=False)

fig.suptitle('WJB Phase 1 — JBB-trained v_harmful(ST) projected onto WildJailbreak', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'wjb_phase1_auroc.png')
plt.show()


### 6b. Summary table — AUROC at best layer per test

For each test, the best AUROC across layers at `t_inst` (the Zhao et al.
harmfulness-encoding position).

In [ ]:
best = (auroc_df[auroc_df['position'] == 't_inst']
        .sort_values('auroc', ascending=False)
        .groupby(['test', 'desc'], as_index=False)
        .first()
        [['test', 'desc', 'layer', 'auroc', 'n_a', 'n_b']])
best['auroc'] = best['auroc'].round(3)
best


## 7. Decision rule

| Observation | Interpretation | Next step |
|---|---|---|
| Test A AUROC ≥ 0.90 and Test B AUROC ≥ 0.90 | Direction transfers cleanly; adversarial framing doesn't break single-turn detection. | Phase 2 (multi-turn WJB) is well-motivated. |
| Test A ≥ 0.9 but Test B drops (≤ 0.75) | Direction transfers *at the prompt-distribution level*, but adversarial framing already displaces single-turn representations. | Phase 2 still interesting — multi-turn displacement would stack on top of this. Frame carefully. |
| Test A < 0.85 | The JBB direction isn't generalizing at all. | Debug v_harmful quality on WJB vanilla before anything else. |
| Test C large (≥ 0.7) at any layer | The direction also separates vanilla from adversarial framing at constant harmfulness — a topic-like confounder. | Report as a caveat; consider subtracting a "framing" direction. |
| Test D close to 0.5 | Adversarial phrasing alone is not encoded along `v_harmful`. Good. | — |